# Task B06: HMDA 2024 Pandas Cleaning Pipeline — `clean_all` and `analytic` (Rulebook v2.3)

**Project:** Decomposing Racial Gaps in Mortgage Denial Rates: A Large-Scale Analysis of the 2024 U.S. HMDA Market  
**Course:** Big Data Analysis (Final Project) — Group 3  
**Milestone:** Task B06 — Chunked Pandas implementation of rules T01–T12, V01–V10, X01–X11, F01–F03 (Plan steps B-3, B-4, B-7, B-8)  
**Companion documents:** `data_cleaning_plan.md`, `data_cleaning_rules.md` (v2.3), `decisions.md` (D-A … D-I), `Coding and documenting conventions.md`

**What this notebook does**
1. **Stage 0 (B-3, global pass):** duplicate key (MD5), lender sizes (X10) and S4 winsorization thresholds — all computed on the full 12.26M rows *before* any filter.
2. **Stage 1 (B-4, main pass):** read the raw Parquet in chunks, apply T → V → X → F rules in rulebook order, and write `B06_clean_all.parquet` (all rows) and `B06_analytic.parquet` (F rules applied) in the same pass.
3. **Stage 2 (B-7, B-8):** rule ledger, golden-number verification against B05, selection waterfalls, and sanity checks.

**Assumptions (labelled per Convention 4.4)**
- `[Verified: B05]` Raw input has 99 HMDA string columns + `row_id` + `bucket`; blanks are `""`, no whitespace padding, 0 unparseable numerics.
- `[Hypothesis]` Same rule semantics will be re-implemented in PySpark in B07 — each ledger row lists the intended PySpark equivalent (Convention 4.6).
- Output files carry the task prefix (`B06_clean_all.parquet`, `B06_analytic.parquet`) per Convention 2.5; the rulebook calls them `clean_all.parquet` / `analytic.parquet`.

**Changes versus the rulebook, to be confirmed by the Human (see end of notebook):** `[ADD]` sample flags `in_s0`, `in_s5_all`, `in_s5_keep_v02`; `[ADD]` `profile_cltv_discrepancy` column for V09; `[CHANGE]` T12 is applied on numeric values right after the T03 cast (equivalent result, simpler than string comparisons).

## Section 1: Configuration & Environment

- **Input:** `input/B04_raw_with_buckets.parquet` (relative to this notebook's folder `B06 - Clean/`).
- **Process:**
  - Resolve paths with `pathlib.Path`; declare constants once (Convention 3.1): chunk size, MD5 separator `chr(31)`, lender-size cut-offs, output paths.
  - `pyarrow` is the Parquet engine. **Arrow** is a columnar in-memory format; we use it for fast trimming and for streaming Parquet reads/writes chunk by chunk so the 12.26M rows never sit in memory at once.
  - `CHUNK_ROWS` and `ENFORCE_GOLDEN` can be overridden by environment variables (`B06_CHUNK_ROWS`, `B06_ENFORCE_GOLDEN`) so the notebook can be smoke-tested on a small sample; the default enforces the B05 golden numbers.
- **Output:** Configured runtime environment.

In [ ]:
# Step 1: Imports
import os, sys, json, time, hashlib, platform, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)   # many column inserts per chunk are intentional

# Step 2: Paths (declared once)
SCRIPT_DIR = Path.cwd()
INPUT_PARQUET = SCRIPT_DIR / "input" / "B04_raw_with_buckets.parquet"
OUTPUT_DIR = SCRIPT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_ALL_PATH = OUTPUT_DIR / "B06_clean_all.parquet"
ANALYTIC_PATH = OUTPUT_DIR / "B06_analytic.parquet"
assert INPUT_PARQUET.exists(), f"Missing input file: {INPUT_PARQUET}"

# Step 3: Constants
CHUNK_ROWS = int(os.environ.get("B06_CHUNK_ROWS", 250_000))
ENFORCE_GOLDEN = os.environ.get("B06_ENFORCE_GOLDEN", "1") == "1"
SEP = chr(31)                        # MD5 key separator (rule V10)
LENDER_CUTS = (500, 5_000, 50_000)   # X10 statutory buckets: <500, <5,000, <50,000, >=50,000
PARQUET_COMPRESSION = "snappy"
print(f"Python {platform.python_version()} | pandas {pd.__version__} | pyarrow {pa.__version__} | numpy {np.__version__}")
print(f"Chunk rows: {CHUNK_ROWS:,} | Enforce golden numbers: {ENFORCE_GOLDEN}")

## Section 1.1: Input Schema & Column Specifications

- **Input:** Parquet metadata of `B04_raw_with_buckets.parquet` (no data is loaded).
- **Process:**
  - Read row count and column names from the Parquet footer; assert 99 HMDA columns plus `row_id` and `bucket`.
  - Declare the column groups used by the rules: the 23 Dictionary `Number` columns (T02 = 14 with `NA`/`Exempt` tokens + 9 others), the integer subset (cast to nullable `Int64`), the 14 Dictionary-documented `1111` columns + `denial_reason_1` (T06 whitelist), the 14 literal-`Exempt` columns (X09), geographic ID columns with expected lengths (T05), and the repeated fields (T09).
  - Assert that every declared column exists in the input.
- **Output:** Column-group constants (`RAW_COLS`, `NUM_COLS`, `T02_COLS`, ...).

In [ ]:
# Step 1: Read Parquet footer only
_pf = pq.ParquetFile(INPUT_PARQUET)
TOTAL_RAW_ROWS = _pf.metadata.num_rows
ALL_COLS = _pf.schema_arrow.names
RAW_COLS = [c for c in ALL_COLS if c not in ("row_id", "bucket")]
assert len(RAW_COLS) == 99 and "row_id" in ALL_COLS and "bucket" in ALL_COLS, "Unexpected input schema"
print(f"Input: {INPUT_PARQUET.name} | rows {TOTAL_RAW_ROWS:,} | columns {len(ALL_COLS)} (99 HMDA + row_id + bucket)")

# Step 2: Numeric column groups (Rule T02 / T03)
T02_COLS = ["income", "property_value", "combined_loan_to_value_ratio", "interest_rate", "rate_spread", "total_loan_costs",
            "total_points_and_fees", "origination_charges", "discount_points", "lender_credits", "loan_term",
            "prepayment_penalty_term", "intro_rate_period", "multifamily_affordable_units"]
NUM_OTHER = ["activity_year", "loan_amount", "tract_population", "tract_minority_population_percent",
             "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage", "tract_owner_occupied_units",
             "tract_one_to_four_family_homes", "tract_median_age_of_housing_units"]
NUM_COLS = T02_COLS + NUM_OTHER
INT_COLS = ["activity_year", "loan_term", "prepayment_penalty_term", "intro_rate_period", "multifamily_affordable_units",
            "tract_population", "tract_owner_occupied_units", "tract_one_to_four_family_homes",
            "tract_median_age_of_housing_units", "ffiec_msa_md_median_family_income"]
T12_TRACT6 = ["tract_population", "tract_minority_population_percent", "tract_owner_occupied_units",
              "tract_one_to_four_family_homes", "tract_median_age_of_housing_units", "tract_to_msa_income_percentage"]
FFIEC_COL = "ffiec_msa_md_median_family_income"
assert len(NUM_COLS) == 23 and set(INT_COLS) <= set(NUM_COLS)

In [ ]:
# Step 1: Categorical column groups (Rules T05, T06, T09, X09)
DOC_1111_COLS = ["aus_1", "manufactured_home_land_property_interest", "balloon_payment", "negative_amortization",
                 "other_nonamortizing_features", "interest_only_payment", "manufactured_home_secured_property_type",
                 "reverse_mortgage", "applicant_credit_score_type", "co_applicant_credit_score_type",
                 "submission_of_application", "initially_payable_to_institution", "business_or_commercial_purpose",
                 "open_end_line_of_credit"]                              # [Verified: Data Dictionary, B05 cell 5]
T06_WHITELIST = DOC_1111_COLS + ["denial_reason_1"]                      # 15 columns; denial_reason_1 is [Hypothesis]
LIT_EXEMPT_COLS = [c for c in T02_COLS if c != "income"] + ["debt_to_income_ratio"]   # 14 literal-'Exempt' columns
ID_SPECS = {"lei": 20, "state_code": 2, "county_code": 5, "census_tract": 11, "derived_msa_md": 5}   # T05 lengths
REPEATED_COLS = ([f"{p}_{k}" for p in ("applicant_ethnicity", "co_applicant_ethnicity", "applicant_race", "co_applicant_race")
                  for k in range(2, 6)] + [f"aus_{k}" for k in range(2, 6)] + [f"denial_reason_{k}" for k in range(2, 5)])

# Step 2: Valid code sets for co-applicant states (X05) and ages (T08)
AGE_VALID = ["<25", "25-34", "35-44", "45-54", "55-64", "65-74", ">74"]
SEX_PRES = ["1", "2", "3", "6"]
RACE_PRES = ["1", "2", "21", "22", "23", "24", "25", "26", "27", "3", "4", "41", "42", "43", "44", "5", "6"]
ETH_PRES = ["1", "11", "12", "13", "14", "2", "3"]

# Step 3: Assert every declared column exists in the input
declared = (set(NUM_COLS) | set(T06_WHITELIST) | set(ID_SPECS) | set(REPEATED_COLS) | set(LIT_EXEMPT_COLS) |
            {"action_taken", "derived_race", "derived_ethnicity", "debt_to_income_ratio", "co_applicant_age", "applicant_age",
             "co_applicant_sex", "co_applicant_race_1", "co_applicant_ethnicity_1", "lien_status", "denial_reason_1",
             "denial_reason_2", "denial_reason_3", "denial_reason_4"})
assert declared <= set(RAW_COLS), f"Declared columns missing from input: {sorted(declared - set(RAW_COLS))}"

## Section 1.2: Golden Numbers from B05 (Verification Targets)

- **Input:** Verified outputs of the B05 DuckDB audit (`B05 - 30 Oct - v5.ipynb`, run 2026-10-31) as recorded in `data_cleaning_rules.md` v2.3 and `decisions.md`.
- **Process:** Store the expected counts in one dictionary, with the B05 cell reference, so Section 7 can compare them to what this Pandas pipeline actually computed. These are **targets for verification only** — no rule reads them.
- **Output:** `GOLDEN` dictionary (metric → expected value). Raw and S0 counts are suffixed `_raw` / `_s0`.

In [ ]:
# Step 1: Golden targets (as of run 2026-10-31; B05 cell references in comments)
GOLDEN = {
    "raw_rows": 12_259_199, "f01_rows": 8_661_779, "f02_rows": 8_170_177, "s0_rows": 8_136_664,       # cell 21
    "s0_denied": 2_003_276, "f03_alone_on_f01": 33_569, "f02_f03_overlap": 56,                         # cells 21, 43
    "f02_1111_in_s0": 250_691, "f03_1111_in_s0": 250_889,                                              # cell 21
    "v01_raw": 0, "v01_s0": 0, "v02_raw": 165_254, "v02_s0": 91_261, "v03_raw": 5_323, "v03_s0": 4_026,  # cell 23
    "v04_raw": 83_443, "v04_s0": 60_717, "v05_raw": 2_893, "v05_s0": 18,                               # cell 23
    "v06_raw": 0, "v07_raw": 0, "v08_raw": 0,                                                          # cells 27-29, B04
    "v10_clusters": 20_229, "v10_extra_copies": 26_755,                                                # rules V10
    "x05_discordant_raw": 79_088, "x05_discordant_s0": 18_230, "x05_all_na_raw": 1_134_611, "x05_all_na_s0": 3_625,  # cell 19
    "x09_full_raw": 276_755, "x09_full_s0": 241_108, "x09_partial_raw": 23_164, "x09_partial_s0": 15_292,           # cell 13
    "dr1_is_1111_raw": 247_789,                                                                        # cell 11
    "t08_applicant_age_8888": 1_463_684, "t08_co_age_8888": 1_178_955, "t08_co_age_9999": 6_666_402,   # plan / cell 15
    "t05_lowercase_na_tract": 1, "t05_non_msa_raw": 1_781_406,                                         # cell 29
    "tract_null_raw": 349_024, "tract_null_s0": 96_065,                                                # cells 29, 33
    "ffiec_no_geo_raw": 293_681, "ffiec_no_geo_s0": 63_462, "median_age_na_code_raw": 204,             # cell 33
    "tract_income_zero_flag_raw": 58_174,                                                              # cell 33
    "s5_all_rows": 8_041_498, "s5_all_denied": 1_944_293,                                              # cell 43
    "s5_keep_v02_rows": 8_132_620, "s5_keep_v02_denied": 1_999_367,                                    # cell 43
}
print(f"{len(GOLDEN)} golden targets loaded.")

## Section 1.3: Reusable Helpers — Artifact Logging, Rule Ledger & F-Mask

- **Input:** None (function definitions).
- **Process:**
  - `save_and_log` / `log_parquet` print path, size and shape for every artifact (Convention 3.8).
  - **Rule ledger (`LEDGER`):** every rule calls `log_mask(key, description, mask, s0, unit)`; the function adds the number of affected rows (or cells) over the **raw** population and over the **S0** population. Counts accumulate across chunks. `WF` collects the F-rule waterfall counts and `T03_AUDIT` the cast audit.
  - `f_masks` implements F01–F03 with **positive set membership** (`.isin`) so NULL/NaN never leak through the filter (Principle 5).
- **Output:** Helper functions and empty accumulators.

In [ ]:
# Step 1: Artifact logging helpers
def save_and_log(df: pd.DataFrame, filename: str) -> Path:
    target = OUTPUT_DIR / filename
    df.to_csv(target, index=False, encoding="utf-8")
    size = target.stat().st_size
    print(f"Saved: {target} | Size: {size/1024.0:.2f} KB ({size:,} bytes) | Shape: {df.shape}")
    return target

def log_parquet(path: Path, n_rows: int, n_cols: int) -> None:
    size = path.stat().st_size
    print(f"Saved: {path} | Size: {size/1024**2:.2f} MB ({size:,} bytes) | Shape: ({n_rows:,}, {n_cols})")

# Step 2: Accumulators shared by all chunks
LEDGER, WF = {}, defaultdict(int)
T03_AUDIT = defaultdict(lambda: {"valid": 0, "missing_total": 0, "unparseable": 0})
TOKENS = defaultdict(int)     # (column, token) -> count of NA / Exempt tokens

def log_mask(key, desc, mask, s0, unit="rows"):
    # Add rows (or cells) flagged by `mask` over raw and over the S0 sub-population
    e = LEDGER.setdefault(key, {"description": desc, "unit": unit, "raw": 0, "s0": 0})
    e["raw"] += int(mask.sum()); e["s0"] += int((mask & s0).sum())

def f_masks(df):
    # F01-F03 sequential masks; positive membership => NULL-safe in Pandas and PySpark
    f01 = df["action_taken"].isin(["1", "2", "3"])
    f02 = f01 & df["business_or_commercial_purpose"].isin(["2", "1111"])
    s0 = f02 & df["reverse_mortgage"].isin(["2", "1111"])
    return f01, f02, s0

## Section 2.1: Stage 0 — Global Computations on the Full Data (Plan B-3)

- **Input:** `B04_raw_with_buckets.parquet`, streamed in chunks.
- **Process:** Three quantities must be computed on **all** rows before any filter or per-chunk logic (a per-chunk version would be wrong):
  - **Duplicate key (V10).** For every row, join all 99 raw columns with `chr(31)` (an invisible control character that cannot occur in the data, so `"ab"+"c"` never collides with `"a"+"bc"`) and hash it with **MD5** (a 128-bit fingerprint; equal rows give equal fingerprints). The 16-byte digest is stored as two `uint64` numbers. Joining is done in Arrow (`binary_join_element_wise`) for speed.
  - **Lender size (X10).** Application count per `lei` over the full 12.26M rows.
  - **S4 thresholds.** p1/p99 of `loan_amount`, `income`, `combined_loan_to_value_ratio`, `property_value`, `loan_to_income`, computed **once on S0** and stored as constants (rulebook §8).
- **Output:** In-memory `LENDER_COUNTS`, `S4_THRESH`, duplicate lookup tables; exported in the next cell.

In [ ]:
# Step 1: Streaming reader and MD5 helpers
def iter_raw_tables(columns=None):
    # Yield the raw Parquet as Arrow tables of CHUNK_ROWS rows
    for batch in pq.ParquetFile(INPUT_PARQUET).iter_batches(batch_size=CHUNK_ROWS, columns=columns):
        yield pa.Table.from_batches([batch])

def md5_halves(tbl):
    # MD5 over the 99 raw columns joined by chr(31); returns an (n, 2) uint64 array
    arrays = [tbl[c].cast(pa.string()) for c in RAW_COLS]
    joined = pc.binary_join_element_wise(*arrays, pa.scalar(SEP), null_handling="replace", null_replacement="")
    digest = b"".join(hashlib.md5(s.encode("utf-8")).digest() for s in joined.to_pylist())
    return np.frombuffer(digest, dtype="<u8").reshape(-1, 2).copy()

# Step 2: S0 numeric values for S4 thresholds
S4_VARS = ["loan_amount", "income", "combined_loan_to_value_ratio", "property_value", "loan_to_income"]
S0_NEEDED = ["lei", "action_taken", "business_or_commercial_purpose", "reverse_mortgage",
             "loan_amount", "income", "combined_loan_to_value_ratio", "property_value"]

def s0_numeric_values(sub):
    # Numeric arrays (non-null) of the S4 variables restricted to S0 rows of one chunk
    trimmed = sub[S0_NEEDED[1:4]].apply(lambda s: s.str.strip())
    s0 = f_masks(trimmed)[2]
    num = {c: pd.to_numeric(sub[c].str.strip(), errors="coerce") for c in S0_NEEDED[4:]}
    num["loan_to_income"] = (num["loan_amount"] / (num["income"] * 1000)).where(num["income"] > 0)
    return {c: num[c][s0].dropna().to_numpy() for c in S4_VARS}

In [ ]:
# Step 1: Duplicate tables from the collected MD5 halves and row ids
def build_dup_tables(halves, ids):
    dk = pd.DataFrame(np.concatenate(halves), columns=["k1", "k2"])
    dk["row_id"] = np.concatenate(ids)
    assert dk["row_id"].is_unique, "row_id must be unique"
    grp = dk.groupby(["k1", "k2"])["row_id"]
    dk["size"], dk["min_id"] = grp.transform("size"), grp.transform("min")
    dups = dk[dk["size"] > 1]
    size_map = dups.set_index("row_id")["size"].astype("int32")
    extra_ids = dups.loc[dups["row_id"] != dups["min_id"], "row_id"].to_numpy()
    n_clusters = int(dups.groupby(["k1", "k2"]).ngroups)
    return size_map, extra_ids, {"clusters": n_clusters, "extra_copies": int(len(extra_ids)), "rows": int(len(dk))}

# Step 2: One streaming pass collecting dup keys, lender counts and S0 numeric values
def global_pass():
    halves, ids, s4 = [], [], defaultdict(list)
    lender = pd.Series(dtype="float64")
    for i, tbl in enumerate(iter_raw_tables()):
        halves.append(md5_halves(tbl))
        ids.append(pd.to_numeric(tbl["row_id"].to_pandas()).to_numpy())
        sub = tbl.select(S0_NEEDED).to_pandas()
        lender = lender.add(sub["lei"].str.strip().value_counts(), fill_value=0)
        for k, v in s0_numeric_values(sub).items():
            s4[k].append(v)
        if i % 10 == 0: print(f"  stage 0: chunk {i} done")
    thresh = {k: dict(zip(("p01", "p99"), map(float, np.quantile(np.concatenate(v), [0.01, 0.99]))),
                      n_values=int(sum(len(x) for x in v))) for k, v in s4.items()}
    return lender.astype("int64"), thresh, *build_dup_tables(halves, ids)

In [ ]:
# Step 1: Run Stage 0 (single pass over the full raw data)
t0 = time.time()
LENDER_COUNTS, S4_THRESH, DUP_SIZE, DUP_EXTRA_IDS, DUP_STATS = global_pass()
assert DUP_STATS["rows"] == TOTAL_RAW_ROWS, "Stage 0 row count mismatch"
print(f"Stage 0 finished in {time.time()-t0:,.0f}s | duplicates: {DUP_STATS['clusters']:,} clusters, {DUP_STATS['extra_copies']:,} extra copies")

# Step 2: Export constants so PySpark (B07) can reuse exactly the same values
with open(OUTPUT_DIR / "B06_s4_winsor_thresholds.json", "w", encoding="utf-8") as f:
    json.dump({"method": "numpy.quantile linear on S0 non-null values", "thresholds": S4_THRESH}, f, indent=2)
lender_df = LENDER_COUNTS.rename_axis("lei").reset_index(name="lender_n_apps")
save_and_log(lender_df, "B06_lender_sizes.csv")
dup_df = DUP_SIZE.rename("dup_group_size").reset_index().assign(flag_dup_extra=lambda d: d["row_id"].isin(DUP_EXTRA_IDS))
save_and_log(dup_df, "B06_dup_table.csv")
display(pd.DataFrame(S4_THRESH).T)

## Section 3.1: Stage 1 — Trim and Raw-Token Status Columns (Rules T01, T02, T05, T06, T10)

- **Input:** One Arrow chunk of the raw table (all strings).
- **Process:**
  - **T01 trim** is done in Arrow (`utf8_trim_whitespace`), which is much faster than Pandas string methods; the number of padded cells is logged (B05 found 0).
  - `""` is **not yet** turned into NULL: the `<col>_status` columns (`value` / `na` / `exempt` / `blank`) must see the raw token first, because after NULL conversion the reason for missingness would be lost. `token_status` classifies each cell case-insensitively.
  - **T02 / T05** status columns are created for the 14 numeric columns and the 5 ID columns; `derived_msa_md == '99999'` gets status `non_msa` `[Hypothesis: non-metropolitan; not in Data Dictionary]`.
  - **T06 / T10** are "keep" rules: `1111` and `denial_reason_1 = 10` stay as strings; the ledger only counts them. **T04 / T11** (categorical codes and `*_above_62` stay strings) need no code.
- **Output:** Pandas DataFrame with status columns added; ledger entries for T01, T02, T05, T06, T10.

In [ ]:
# Step 1: Arrow-side trim (T01); returns a Pandas frame and the number of padded cells
def trim_table(tbl):
    cols, padded = {}, 0
    for name in tbl.column_names:
        arr = tbl[name]
        if name in RAW_COLS:
            arr = arr.cast(pa.string())
            trimmed = pc.utf8_trim_whitespace(arr)
            padded += pc.sum(pc.fill_null(pc.not_equal(arr, trimmed), False)).as_py() or 0
            arr = trimmed
        cols[name] = arr
    return pa.table(cols).to_pandas(), int(padded)

# Step 2: Case-insensitive token classifier used by every <col>_status column
def token_status(s):
    up = s.str.upper()
    vals = np.select([s.isna() | (s == ""), up == "NA", up == "EXEMPT"], ["blank", "na", "exempt"], default="value")
    return pd.Series(vals, index=s.index)

def status_col(c):
    return "dti_status" if c == "debt_to_income_ratio" else f"{c}_status"

In [ ]:
# Step 1: T02 / T05 status columns from raw tokens
def rule_status_cols(df, s0):
    for c in T02_COLS + list(ID_SPECS):
        st = token_status(df[c])
        df[f"{c}_status"] = st
        for tok in ("na", "exempt"):
            TOKENS[(c, tok)] += int((st == tok).sum())
        key = "T02:na_exempt_tokens" if c in T02_COLS else "T05:na_exempt_tokens"
        log_mask(key, "NA/Exempt tokens converted to NULL (cells)", st.isin(["na", "exempt"]), s0, "cells")
    non_msa = df["derived_msa_md"] == "99999"
    df.loc[non_msa, "derived_msa_md_status"] = "non_msa"
    log_mask("T05:non_msa_99999", "derived_msa_md = 99999 -> status non_msa", non_msa, s0)
    log_mask("T05:lowercase_na_census_tract", "census_tract lowercase 'na'", df["census_tract"] == "na", s0)

# Step 2: T06 / T10 keep-rules (count only)
def rule_keep_codes(df, s0):
    for c in T06_WHITELIST:
        log_mask("T06:code_1111_kept", "Code 1111 kept as category Exempt (cells, 15 whitelist cols)", df[c] == "1111", s0, "cells")
    log_mask("T10:denial_reason_1_code_10_kept", "denial_reason_1 = 10 kept as valid category", df["denial_reason_1"] == "10", s0)

## Section 3.2: Stage 1 — DTI Mapping (T07) and Age Special Codes (T08)

- **Input:** DataFrame with raw tokens still present.
- **Process:**
  - **T07:** strip `%`, then map through an **explicit lookup with no pass-through**: any value outside the lookup raises an `AssertionError` (B05 cell 31: all 21 distinct values map). Creates `dti_bin_fine`, `dti_bin_coarse`, `dti_point` (36–49 only) and `dti_status`; `NA`/`Exempt` give NULL bins.
  - **T08:** `8888` ("Not applicable") → NULL with status `na`; for the co-applicant `9999` ("No co-applicant") → NULL with status `no_coapp`. Anything outside the valid age brackets is marked `invalid` (expected 0). `has_coapplicant` is **not** set here (see X05).
- **Output:** `debt_to_income_ratio` (cleaned), `dti_*` columns, `applicant_age(_status)`, `co_applicant_age(_status)`.

In [ ]:
# Step 1: Explicit DTI lookup tables (no pass-through)
DTI_POINTS = [str(i) for i in range(36, 50)]
DTI_FINE = {"<20": "<20", "20-<30": "20-<30", "30-<36": "30-<36", "50-60": "50-60", ">60": ">60"}
DTI_COARSE = dict(DTI_FINE)
for _i in range(36, 50):
    DTI_FINE[str(_i)] = str(_i)
    DTI_COARSE[str(_i)] = "36-40" if _i <= 40 else ("41-45" if _i <= 45 else "46-49")

# Step 2: Rule T07
def rule_t07(df, s0):
    st = token_status(df["debt_to_income_ratio"])
    key = df["debt_to_income_ratio"].str.replace("%", "", regex=False)
    is_val = st == "value"
    unknown = is_val & ~key.isin(list(DTI_FINE))
    assert not unknown.any(), f"Unmapped DTI values: {key[unknown].unique()[:5]}"
    for tok in ("na", "exempt"):
        TOKENS[("debt_to_income_ratio", tok)] += int((st == tok).sum())
    clean = key.where(is_val)
    df["dti_status"], df["debt_to_income_ratio"] = st, clean
    df["dti_bin_fine"], df["dti_bin_coarse"] = clean.map(DTI_FINE), clean.map(DTI_COARSE)
    df["dti_point"] = pd.to_numeric(clean.where(clean.isin(DTI_POINTS)), errors="coerce")
    log_mask("T07:dti_mapped", "DTI value mapped to fine/coarse bins", is_val, s0)
    log_mask("T07:dti_na_exempt", "DTI NA/Exempt -> NULL bins", st.isin(["na", "exempt"]), s0)

In [ ]:
# Step 1: Rule T08 for applicant_age and co_applicant_age
def rule_t08(df, s0):
    for col in ("applicant_age", "co_applicant_age"):
        s = df[col]
        conds = [s.isna() | (s == ""), s.isin(AGE_VALID), s == "8888"]
        choices = ["blank", "value", "na"]
        if col == "co_applicant_age":
            conds.append(s == "9999"); choices.append("no_coapp")
        st = pd.Series(np.select(conds, choices, default="invalid"), index=s.index)
        df[f"{col}_status"], df[col] = st, s.where(st == "value")
        for tag in ("na", "no_coapp", "invalid"):
            log_mask(f"T08:{col}_{tag}", f"{col} status {tag} -> NULL", st == tag, s0)

## Section 3.3: Stage 1 — Co-Applicant 4-State Classification (Rule X05)

- **Input:** DataFrame with raw co-applicant tokens (must run **before** T08 / NULL conversion).
- **Process:**
  - Map each of `co_applicant_age`, `_sex`, `_race_1`, `_ethnicity_1` with **explicit valid sets** to `Present` / `No_Coapp` / `NA` / `Invalid`. Blank or NULL in `race_1` and `ethnicity_1` counts as `NA` `[Hypothesis: unclassified or free-form text co-applicant]`; blank in age or sex is `Invalid`.
  - Precedence **Present > No_Coapp > NA**: `has_coapplicant = 1` if any field is Present; else `0` if any is No_Coapp; else NULL. `flag_coapp_discordant` is true when the four states are not all equal.
- **Output:** `has_coapplicant` (nullable `Int8`), `flag_coapp_discordant`; ledger entries `X05:*` (including the `Invalid` count, expected 0).

In [ ]:
# Step 1: Explicit 4-state mapper
def state4(s, present, no_coapp, na_tokens, blank_is_na):
    na_cond = s.isin(na_tokens) | (blank_is_na & (s.isna() | (s == "")))
    return np.select([s.isin(present), s.isin(no_coapp), na_cond], ["Present", "No_Coapp", "NA"], default="Invalid")

# Step 2: Rule X05
def rule_x05(df, s0):
    st = pd.DataFrame({
        "age": state4(df["co_applicant_age"], AGE_VALID, ["9999"], ["8888"], False),
        "sex": state4(df["co_applicant_sex"], SEX_PRES, ["5"], ["4"], False),
        "race": state4(df["co_applicant_race_1"], RACE_PRES, ["8"], ["7"], True),
        "eth": state4(df["co_applicant_ethnicity_1"], ETH_PRES, ["5"], ["4"], True)}, index=df.index)
    any_p, any_n = (st == "Present").any(axis=1), (st == "No_Coapp").any(axis=1)
    codes = np.select([any_p, any_n], [1, 0], default=-1)
    df["has_coapplicant"] = pd.Series(codes, index=df.index).astype("Int8").mask(codes == -1)
    discordant = st.nunique(axis=1) > 1
    df["flag_coapp_discordant"] = discordant.to_numpy()
    log_mask("X05:discordant", "flag_coapp_discordant (4 states not all equal)", discordant, s0)
    log_mask("X05:all_na", "All 4 states NA -> has_coapplicant NULL", (st == "NA").all(axis=1), s0)
    log_mask("X05:invalid", "Any state Invalid (expected 0)", (st == "Invalid").any(axis=1), s0)
    log_mask("X05:all_na_purchased", "All-NA rows among purchased loans (action_taken 6)",
             (st == "NA").all(axis=1) & (df["action_taken"] == "6"), s0)

## Section 3.4: Stage 1 — NULL Conversion (T01, T09, T02/T05 tokens) and Numeric Cast (T03)

- **Input:** DataFrame with status columns and the raw tokens.
- **Process:**
  - **T01 / T09:** every `""` in the 99 raw columns becomes NULL (structural absence for the repeated fields `*_2..5`). `NA` is **not** globally nulled: T11 and other categorical columns keep `NA` as a legitimate label.
  - **T02 / T05:** `NA`/`Exempt` tokens in the 14 numeric and 5 ID columns become NULL (status columns already recorded the reason). `1111`, `8888`, `9999` stay ordinary numbers in numeric columns.
  - **T03:** explicit cast of the 23 Dictionary `Number` columns with `pd.to_numeric(errors="coerce")`. The audit records `valid`, `missing_total` and `unparseable` per column; `valid + missing_total + unparseable` must equal the chunk size and `unparseable` must be 0.
  - **T05 length check:** non-NULL IDs must have length 20 / 2 / 5 / 11 / 5; **no zero padding** is applied.
- **Output:** NULL-normalised string columns and 23 float64 numeric columns (final integer dtypes are applied at write time).

In [ ]:
# Step 1: T01 / T09 blank -> NULL and T02 / T05 token -> NULL
def rule_nullify(df, s0):
    for c in RAW_COLS:
        m = df[c] == ""
        if m.any():
            df[c] = df[c].mask(m)
        key = "T09:blank_to_null" if c in REPEATED_COLS else "T01:blank_to_null"
        log_mask(key, "Blank string converted to NULL (cells)", m, s0, "cells")
    for c in T02_COLS + list(ID_SPECS):
        m = df[f"{c}_status"].isin(["na", "exempt"])
        df[c] = df[c].mask(m)

# Step 2: T05 length check on non-NULL IDs
def rule_id_checks(df, s0):
    for c, length in ID_SPECS.items():
        bad = df[c].notna() & (df[c].str.len() != length)
        log_mask("T05:length_violations", "Non-NULL ID with unexpected length (cells)", bad, s0, "cells")

In [ ]:
# Step 1: Rule T03 cast with per-column audit
def rule_cast(df):
    n = len(df)
    for c in NUM_COLS:
        raw = df[c]
        num = pd.to_numeric(raw, errors="coerce")
        a = T03_AUDIT[c]
        a["valid"] += int(num.notna().sum())
        a["missing_total"] += int(raw.isna().sum())
        a["unparseable"] += int((num.isna() & raw.notna()).sum())
        assert int(num.notna().sum()) + int(raw.isna().sum()) + int((num.isna() & raw.notna()).sum()) == n, f"T03 audit mismatch on {c}"
        if c in INT_COLS:
            frac = num.notna() & (num != num.round())
            assert not frac.any(), f"{c} has non-integer values {num[frac].unique()[:3]}; move it from INT_COLS to float"
        df[c] = num.astype("float64")

## Section 3.5: Stage 1 — Census Per-Field Rule T12 (Decisions D-G, D-H)

- **Input:** DataFrame with numeric tract fields (after the T03 cast) and NULL-normalised `census_tract`, `county_code`.
- **Process:** Each field is treated on its own evidence (B05 cell 33). Status values: `value`, `no_tract`, `no_geo`, `na_code`.
  1. Six tract fields: value `0` on rows whose `census_tract` is NULL → NULL, status `no_tract`. Any **non-zero** value on a tract-NA row is kept and counted as an anomaly (expected 0).
  2. `ffiec_msa_md_median_family_income`: `0` where both `county_code` and `census_tract` are NULL → NULL, status `no_geo`. Positive values on tract-NA rows are **preserved**. Zeros elsewhere are kept and counted as an anomaly (expected 0).
  3. `tract_median_age_of_housing_units`: `-1` → NULL, status `na_code`. Zeros on valid tracts are kept.
  4. `tract_to_msa_income_percentage`: zeros on valid tracts stay `0` with `flag_tract_income_zero` (RQ3 runs with and without them).
  5. `tract_minority_population_percent` must lie in [0, 100]; violations are counted (expected 0).
- **Output:** 7 `<col>_status` columns, `flag_tract_income_zero`, ledger entries `T12*`.

In [ ]:
# Step 1: Items 1, 3 and 5 (six tract fields)
def rule_t12_tract_fields(df, s0, tract_na):
    for c in T12_TRACT6:
        st = pd.Series(np.where(df[c].notna(), "value", "blank"), index=df.index)
        zero = tract_na & (df[c] == 0)
        log_mask(f"T12.1:{c}", f"{c}: zero on tract-NA row -> NULL (no_tract)", zero, s0)
        log_mask("T12.1:anomaly_nonzero_on_tract_na", "Non-zero tract value on tract-NA row (expected 0)",
                 tract_na & df[c].notna() & (df[c] != 0), s0, "cells")
        df.loc[zero, c], st[zero] = np.nan, "no_tract"
        if c == "tract_median_age_of_housing_units":
            neg = df[c] == -1
            log_mask("T12.3:median_age_na_code", "tract_median_age_of_housing_units = -1 -> NULL (na_code)", neg, s0)
            df.loc[neg, c], st[neg] = np.nan, "na_code"
        df[f"{c}_status"] = st
    out = df["tract_minority_population_percent"].notna() & ~df["tract_minority_population_percent"].between(0, 100)
    log_mask("T12.5:minority_out_of_range", "tract_minority_population_percent outside [0, 100] (expected 0)", out, s0)

# Step 2: Items 2 and 4 (FFIEC income, tract-to-MSA income zero flag)
def rule_t12_income_fields(df, s0, tract_na, county_na):
    no_geo = (df[FFIEC_COL] == 0) & tract_na & county_na
    log_mask("T12.2:ffiec_zero_no_geo", "FFIEC income 0 with county and tract both NULL -> NULL (no_geo)", no_geo, s0)
    log_mask("T12.2:anomaly_ffiec_zero_elsewhere", "FFIEC income 0 outside no-geo rows (expected 0)",
             (df[FFIEC_COL] == 0) & ~(tract_na & county_na), s0)
    st = pd.Series(np.where(df[FFIEC_COL].notna(), "value", "blank"), index=df.index)
    df.loc[no_geo, FFIEC_COL], st[no_geo] = np.nan, "no_geo"
    df[f"{FFIEC_COL}_status"] = st
    df["flag_tract_income_zero"] = (df["tract_to_msa_income_percentage"] == 0).to_numpy()
    log_mask("T12.4:tract_income_zero_flag", "Zero tract_to_msa_income_percentage kept on valid tract (flagged)",
             df["flag_tract_income_zero"], s0)

def rule_t12(df, s0):
    tract_na, county_na = df["census_tract"].isna(), df["county_code"].isna()
    log_mask("T12.0:tract_null", "census_tract is NULL", tract_na, s0)
    rule_t12_tract_fields(df, s0, tract_na)
    rule_t12_income_fields(df, s0, tract_na, county_na)

## Section 3.6: Stage 1 — Validity Flags V01–V08 (flag, never delete)

- **Input:** Cleaned numeric and ID columns.
- **Process:** Each rule creates a boolean flag; comparisons against NULL are `False` in Pandas (the PySpark version wraps conditions in `coalesce(..., False)` to match). Rows are **never removed** (Principle 2).
  - V01 `loan_amount <= 0`; V02 `income <= 0` with sub-flags zero/negative; V03 CLTV `<= 0` or `> 200`; V04 `interest_rate <= 0` or `> 25` with sub-flags; V05 `loan_term` outside [1, 480].
  - V06 `census_tract` fails `^[0-9]{11}$` (after T05); V07 **guarded** county/tract prefix mismatch (both non-NULL, format valid) — the guard removes the Pandas-vs-Spark NULL divergence found in B05 cell 29.
  - V08 denial-reason inconsistency: real reason 1–9 with `action_taken` not in {3, 7}, or `action_taken` in {3, 7} with reason `10`/blank (`1111` = Exempt, never a real reason).
- **Output:** Boolean `flag_*` columns and ledger entries `V01`–`V08`.

In [ ]:
# Step 1: V01-V05 numeric range flags
def rule_v_numeric(df, s0):
    inc, rate, cltv = df["income"], df["interest_rate"], df["combined_loan_to_value_ratio"]
    flags = {
        "flag_loan_amount_invalid": df["loan_amount"] <= 0,
        "flag_income_nonpositive": inc <= 0, "flag_income_zero": inc == 0, "flag_income_negative": inc < 0,
        "flag_cltv_range": (cltv <= 0) | (cltv > 200),
        "flag_rate_le0": rate <= 0, "flag_rate_gt25": rate > 25, "flag_rate_range": (rate <= 0) | (rate > 25),
        "flag_term_range": (df["loan_term"] < 1) | (df["loan_term"] > 480)}
    keys = {"flag_loan_amount_invalid": "V01", "flag_income_nonpositive": "V02", "flag_income_zero": "V02:zero",
            "flag_income_negative": "V02:negative", "flag_cltv_range": "V03", "flag_rate_range": "V04",
            "flag_rate_le0": "V04:le0", "flag_rate_gt25": "V04:gt25", "flag_term_range": "V05"}
    for name, mask in flags.items():
        df[name] = mask.to_numpy()
        log_mask(keys[name], f"{name}", mask, s0)

In [ ]:
# Step 1: V06-V08 identifier and denial-reason consistency flags
def rule_v_ids_and_reasons(df, s0):
    tract, county = df["census_tract"], df["county_code"]
    bad_fmt = tract.notna() & ~tract.str.fullmatch(r"[0-9]{11}", na=False)
    df["flag_id_format"] = bad_fmt.to_numpy()
    mismatch = county.notna() & tract.notna() & ~bad_fmt & (county.str[:5] != tract.str[:5])
    df["flag_geo_mismatch"] = mismatch.to_numpy()
    reasons = df[["denial_reason_1", "denial_reason_2", "denial_reason_3", "denial_reason_4"]]
    real_any = reasons.isin(list("123456789")).any(axis=1)
    decided_no = df["action_taken"].isin(["3", "7"])
    dr1 = df["denial_reason_1"]
    inconsistent = (real_any & ~decided_no) | (decided_no & (dr1.isna() | (dr1 == "10")))
    df["flag_denial_inconsistent"] = inconsistent.to_numpy()
    for key, name, m in (("V06", "flag_id_format", bad_fmt), ("V07", "flag_geo_mismatch", mismatch),
                         ("V08", "flag_denial_inconsistent", inconsistent)):
        log_mask(key, name, m, s0)

## Section 3.7: Stage 1 — Feature Engineering X01–X11, Diagnostic V09 and Duplicates V10

- **Input:** Cleaned DataFrame, `LENDER_COUNTS`, `DUP_SIZE`, `DUP_EXTRA_IDS` from Stage 0.
- **Process:**
  - X01 `is_denied` (3 → 1; 1, 2 → 0; else NULL), X02 `in_decision_sample`, X03/X04 `race_group` / `ethnicity_group` (all 9 race groups kept), X05 done above.
  - X06 `loan_to_income = loan_amount / (income * 1000)`, NULL if income ≤ 0 or NULL; X07 `ltv_calc = loan_amount / property_value * 100`; X08 `log1p` of loan amount and income (NULL if input ≤ 0).
  - X09 on the **28 documented fields only** (14 literal `Exempt` + 14 documented `1111`): `n_exempt_28`, `is_exempt_filer`, `is_partial_exempt`; `flag_dr1_is_1111` tracked separately `[Hypothesis]`.
  - X10 `lender_n_apps` and `lender_size_bucket` (Small/Community < 500, Medium 500–4,999, Large 5,000–49,999, Mega ≥ 50,000) from the full-data counts; X11 `fin_complete`.
  - **V09 (diagnostic only):** `profile_cltv_discrepancy = CLTV − ltv_calc` (no exclusion). **V10:** `dup_group_size`, `flag_dup_extra` from Stage 0.
- **Output:** All X columns, `profile_cltv_discrepancy`, `dup_group_size`, `flag_dup_extra`.

In [ ]:
# Step 1: X01-X08, X11 and V09
def rule_x_core(df, s0):
    act = df["action_taken"]
    den = np.select([act == "3", act.isin(["1", "2"])], [1, 0], default=-1)
    df["is_denied"] = pd.Series(den, index=df.index).astype("Int8").mask(den == -1)
    df["in_decision_sample"] = act.isin(["1", "2", "3"]).to_numpy()
    df["race_group"], df["ethnicity_group"] = df["derived_race"], df["derived_ethnicity"]
    la, inc, pv = df["loan_amount"], df["income"], df["property_value"]
    df["loan_to_income"] = (la / (inc * 1000)).where(inc > 0)
    df["ltv_calc"] = (la / pv * 100).where(pv > 0)
    df["log_loan_amount"], df["log_income"] = np.log1p(la.where(la > 0)), np.log1p(inc.where(inc > 0))
    df["fin_complete"] = df[["dti_bin_fine", "combined_loan_to_value_ratio", "income", "loan_amount"]].notna().all(axis=1).to_numpy()
    cltv = df["combined_loan_to_value_ratio"]
    df["profile_cltv_discrepancy"] = (cltv - df["ltv_calc"]).where((cltv > 0) & df["ltv_calc"].notna())
    log_mask("V09:profile_rows", "Rows with a CLTV-vs-LTV discrepancy profile value (diagnostic)", df["profile_cltv_discrepancy"].notna(), s0)

In [ ]:
# Step 1: X09 exempt-filer features on the 28 documented fields
def rule_x09(df, s0):
    lit = sum((df[status_col(c)] == "exempt").astype(int) for c in LIT_EXEMPT_COLS)
    doc = sum((df[c] == "1111").astype(int) for c in DOC_1111_COLS)
    n28 = lit + doc
    assert n28.between(0, 28).all(), "n_exempt_28 outside 0..28"
    df["n_exempt_28"] = n28.astype("Int8")
    df["is_exempt_filer"] = (n28 == 28).to_numpy()
    df["is_partial_exempt"] = n28.between(1, 27).to_numpy()
    df["flag_dr1_is_1111"] = (df["denial_reason_1"] == "1111").to_numpy()
    log_mask("X09:full_exempt", "is_exempt_filer (n_exempt_28 == 28)", n28 == 28, s0)
    log_mask("X09:partial_exempt", "is_partial_exempt (1 <= n <= 27)", n28.between(1, 27), s0)
    log_mask("X09:dr1_is_1111", "flag_dr1_is_1111", df["flag_dr1_is_1111"], s0)

# Step 2: X10 lender size and V10 duplicates (lookups from Stage 0)
def rule_x10_v10(df, s0):
    n = df["lei"].str.strip().map(LENDER_COUNTS)
    c1, c2, c3 = LENDER_CUTS
    bucket = np.select([n < c1, n < c2, n < c3, n >= c3], ["Small/Community", "Medium", "Large", "Mega"], default="")
    df["lender_n_apps"] = n.astype("Int64")
    df["lender_size_bucket"] = pd.Series(bucket, index=df.index).where(bucket != "")
    df["dup_group_size"] = df["row_id"].map(DUP_SIZE).fillna(1).astype("int32")
    df["flag_dup_extra"] = df["row_id"].isin(DUP_EXTRA_IDS).to_numpy()
    log_mask("V10:flag_dup_extra", "Extra copies of duplicated rows (first = smallest row_id)", df["flag_dup_extra"], s0)
    log_mask("V10:rows_in_dup_clusters", "Rows belonging to a duplicate cluster", df["dup_group_size"] > 1, s0)

## Section 3.8: Stage 1 — Sample Flags `[ADD]`, Orchestrator and Dtype Finalisation

- **Input:** Cleaned chunk with all V/X columns.
- **Process:**
  - `[ADD]` `in_s0`, `in_s5_all` (exclude V01, V02, V03, V05; **keep V04**, Decisions D-B/D-C) and `in_s5_keep_v02` (exclude V01, V03, V05). They let B08 and the RQ steps select samples without recomputing rules, and let this notebook verify the S5 golden numbers.
  - `clean_chunk` runs all rules in the fixed order **trim → F counts → status → keep-rules → T07 → X05 → T08 → NULL conversion → T03 cast → T05 checks → T12 → V → X → V10 → samples**, and asserts that the row count of the chunk is unchanged.
  - `finalize_dtypes` forces a **stable dtype per column** (nullable `Int64`/`Int8`, `float64`, `bool`, `string`) so every chunk writes an identical Parquet schema.
- **Output:** `clean_chunk(tbl)` → `(DataFrame, S0 mask)`; dtype map functions.

In [ ]:
# Step 1: Sample flags and S5 counts
def rule_samples(df, s0):
    s5_all = s0 & ~(df["flag_loan_amount_invalid"] | df["flag_income_nonpositive"] | df["flag_cltv_range"] | df["flag_term_range"])
    s5_keep = s0 & ~(df["flag_loan_amount_invalid"] | df["flag_cltv_range"] | df["flag_term_range"])
    df["in_s0"], df["in_s5_all"], df["in_s5_keep_v02"] = s0.to_numpy(), s5_all.to_numpy(), s5_keep.to_numpy()
    den = df["is_denied"] == 1
    for name, m in (("s0", s0), ("s5_all", s5_all), ("s5_keep_v02", s5_keep)):
        WF[f"{name}_sample_rows"] += int(m.sum()); WF[f"{name}_sample_denied"] += int((m & den).sum())

# Step 2: Dtype finalisation (stable Parquet schema across chunks)
INT64_SET = set(INT_COLS) | {"lender_n_apps"}
FLOAT_SET = (set(NUM_COLS) - set(INT_COLS)) | {"dti_point", "loan_to_income", "ltv_calc", "log_loan_amount", "log_income",
                                               "profile_cltv_discrepancy"}
INT8_SET = {"is_denied", "has_coapplicant", "n_exempt_28"}
BOOL_SET = {"in_decision_sample", "is_exempt_filer", "is_partial_exempt", "fin_complete", "in_s0", "in_s5_all", "in_s5_keep_v02"}

def finalize_dtypes(df):
    for c in df.columns:
        if c in INT64_SET: df[c] = df[c].astype("Int64")
        elif c in INT8_SET: df[c] = df[c].astype("Int8")
        elif c in FLOAT_SET: df[c] = df[c].astype("float64")
        elif c in BOOL_SET or c.startswith("flag_"): df[c] = df[c].astype(bool)
        elif c == "dup_group_size": df[c] = df[c].astype("int32")
        elif c in ("row_id", "bucket"): df[c] = pd.to_numeric(df[c]).astype("int64")
        else: df[c] = df[c].astype("string")
    return df

In [ ]:
# Step 1: F-rule waterfall counts for one chunk
def log_waterfall(df, f01, f02, s0):
    WF["raw"] += len(df); WF["f01"] += int(f01.sum()); WF["f02"] += int(f02.sum()); WF["s0"] += int(s0.sum())
    f03_alone = f01 & (df["reverse_mortgage"] == "1")
    WF["f03_alone_on_f01"] += int(f03_alone.sum())
    WF["f02_f03_overlap"] += int((f03_alone & (df["business_or_commercial_purpose"] == "1")).sum())
    WF["f02_1111_in_s0"] += int((s0 & (df["business_or_commercial_purpose"] == "1111")).sum())
    WF["f03_1111_in_s0"] += int((s0 & (df["reverse_mortgage"] == "1111")).sum())

# Step 2: Orchestrator: apply every rule to one raw Arrow chunk
def clean_chunk(tbl):
    df, padded = trim_table(tbl)
    n = len(df)
    df["row_id"] = pd.to_numeric(df["row_id"])
    f01, f02, s0 = f_masks(df)
    log_waterfall(df, f01, f02, s0)
    _e = LEDGER.setdefault("T01:trim", {"description": "Padded cells trimmed (cells)", "unit": "cells", "raw": 0, "s0": 0})
    _e["raw"] += padded
    rule_status_cols(df, s0); rule_keep_codes(df, s0); rule_t07(df, s0); rule_x05(df, s0); rule_t08(df, s0)
    rule_nullify(df, s0); rule_cast(df); rule_id_checks(df, s0); rule_t12(df, s0)
    rule_v_numeric(df, s0); rule_v_ids_and_reasons(df, s0)
    rule_x_core(df, s0); rule_x09(df, s0); rule_x10_v10(df, s0); rule_samples(df, s0)
    assert len(df) == n, "Row count changed inside a T/V/X rule"
    return finalize_dtypes(df), s0

## Section 4: Run the Chunked Pipeline — Write `clean_all` and `analytic` in One Pass (Plan B-4, B-7)

- **Input:** Raw Parquet chunks; all rule functions above; Stage 0 lookups.
- **Process:**
  - For every chunk call `clean_chunk`, then write the **whole chunk** to `B06_clean_all.parquet` and the **S0 rows** to `B06_analytic.parquet`. A **Parquet writer** (`pyarrow.parquet.ParquetWriter`) appends row groups to one open file, so the output is a single unpartitioned file per Decision D-E and never needs all rows in memory.
  - The schema is fixed from the first chunk; later chunks are cast to it (`finalize_dtypes` guarantees matching types) and must have the identical column set.
  - Row counts are printed per chunk and reconciled at the end (Convention 4.2).
- **Output:** `output/B06_clean_all.parquet`, `output/B06_analytic.parquet`; filled `LEDGER`, `WF`, `T03_AUDIT`, `TOKENS`.

In [ ]:
# Step 1: Streaming write of both layers
def run_pipeline():
    w_all = w_an = schema = col_order = None
    n_all = n_an = 0
    t0 = time.time()
    try:
        for i, tbl in enumerate(iter_raw_tables()):
            df, s0 = clean_chunk(tbl)
            col_order = col_order or list(df.columns)
            assert set(df.columns) == set(col_order), "Column set changed between chunks"
            out = pa.Table.from_pandas(df[col_order], preserve_index=False)
            if w_all is None:
                schema = out.schema
                w_all = pq.ParquetWriter(CLEAN_ALL_PATH, schema, compression=PARQUET_COMPRESSION)
                w_an = pq.ParquetWriter(ANALYTIC_PATH, schema, compression=PARQUET_COMPRESSION)
            out = out.cast(schema)
            part = out.filter(pa.array(s0.to_numpy()))
            w_all.write_table(out); w_an.write_table(part)
            n_all += out.num_rows; n_an += part.num_rows
            print(f"  chunk {i:>3}: {out.num_rows:>9,} rows -> clean_all | {part.num_rows:>9,} -> analytic | {time.time()-t0:>7,.0f}s")
    finally:
        for w in (w_all, w_an):
            if w is not None: w.close()
    return {"rows_clean_all": n_all, "rows_analytic": n_an, "n_cols": len(col_order), "seconds": time.time() - t0}

# Step 2: Execute and log artifacts
PIPE = run_pipeline()
log_parquet(CLEAN_ALL_PATH, PIPE["rows_clean_all"], PIPE["n_cols"])
log_parquet(ANALYTIC_PATH, PIPE["rows_analytic"], PIPE["n_cols"])
assert PIPE["rows_clean_all"] == TOTAL_RAW_ROWS == WF["raw"], "clean_all must keep every raw row"
assert PIPE["rows_analytic"] == WF["s0"], "analytic must equal S0"

## Section 5.1: T03 Cast Audit and Sentinel Token Counts

- **Input:** `T03_AUDIT` and `TOKENS` accumulated during the run.
- **Process:**
  - Rule T03 audit per numeric column: `valid + missing_total + unparseable = total rows` and `unparseable = 0`.
  - Token counts: number of `NA` and `Exempt` cells per column. **Check:** exactly the 14 literal-`Exempt` columns of X09 carry `Exempt` tokens (and `income` carries none) — this confirms the column list used for `n_exempt_28`.
- **Output:** `output/B06_t03_cast_audit.csv`, `output/B06_sentinel_token_counts.csv`.

In [ ]:
# Step 1: T03 audit table and assertions
t03_df = pd.DataFrame([{"column_name": c, **T03_AUDIT[c]} for c in NUM_COLS])
t03_df["total"] = t03_df[["valid", "missing_total", "unparseable"]].sum(axis=1)
assert (t03_df["total"] == TOTAL_RAW_ROWS).all(), "T03: valid + missing + unparseable != total rows"
assert (t03_df["unparseable"] == 0).all(), "T03: unparseable numeric strings found"
save_and_log(t03_df, "B06_t03_cast_audit.csv")

# Step 2: Token counts and X09 column-list check
tok_df = pd.DataFrame([{"column_name": c, "token": t, "cells": n} for (c, t), n in sorted(TOKENS.items())])
tok_pivot = tok_df.pivot(index="column_name", columns="token", values="cells").fillna(0).astype(int).reset_index()
save_and_log(tok_pivot, "B06_sentinel_token_counts.csv")
exempt_cols = set(tok_pivot.loc[tok_pivot["exempt"] > 0, "column_name"])
assert exempt_cols == set(LIT_EXEMPT_COLS), f"Literal-Exempt columns differ from X09 list: {exempt_cols ^ set(LIT_EXEMPT_COLS)}"
print(f"Literal-Exempt columns confirmed: {len(exempt_cols)} (income excluded: {'income' not in exempt_cols})")

## Section 5.2: Rule Ledger with Pandas and PySpark Equivalents (Plan B-8, Convention 4.6)

- **Input:** `LEDGER`, `WF`, plus a documentation dictionary `RULE_IMPL` giving, for every rule, the Pandas implementation used here and the intended PySpark equivalent for B07.
- **Process:** Build one table with the rule key, description, unit (rows or cells), affected count and percentage over **raw** and **S0**, and both implementations. The three F rules are added from the sequential waterfall (rows excluded).
- **Output:** `output/B06_rule_ledger.csv`.

In [ ]:
# Step 1: Pandas / PySpark implementation notes per rule
RULE_IMPL = {
    "T01": ("pyarrow utf8_trim_whitespace; '' -> NaN via Series.mask", "F.trim(c); F.when(c == '', None).otherwise(c)"),
    "T02": ("token_status() + mask NA/Exempt; <col>_status", "F.when(F.upper(c).isin('NA','EXEMPT'), None); status via F.when chain"),
    "T05": ("token_status() + mask; derived_msa_md 99999 -> non_msa; str.len check", "same as T02 + F.length(c) != L (no lpad)"),
    "T06": ("keep as string, count only", "no-op (columns stay string)"),
    "T07": ("explicit dict lookup via Series.map + assert no unmapped", "F.create_map(...) lookup + assert on unmapped count"),
    "T08": ("np.select -> status; where(status == 'value')", "F.when(c == '8888', None) / F.when(c == '9999', None)"),
    "T09": ("'' -> NaN on repeated fields", "same as T01 on *_2..5 columns"),
    "T10": ("keep code 10 as string", "no-op"),
    "T12": ("per-field masks after T03 cast, status columns", "F.when(tract.isNull() & (c == 0), None) per field"),
    "V01": ("loan_amount <= 0", "F.coalesce(F.col('loan_amount') <= 0, F.lit(False))"),
    "V02": ("income <= 0 (+ zero / negative)", "F.coalesce(F.col('income') <= 0, F.lit(False))"),
    "V03": ("cltv <= 0 | cltv > 200", "coalesce((cltv <= 0) | (cltv > 200), False)"),
    "V04": ("rate <= 0 | rate > 25 (flag only, never excluded)", "coalesce((rate <= 0) | (rate > 25), False)"),
    "V05": ("loan_term < 1 | > 480", "coalesce((term < 1) | (term > 480), False)"),
    "V06": ("notna & ~str.fullmatch('[0-9]{11}')", "c.isNotNull() & ~c.rlike('^[0-9]{11}$')"),
    "V07": ("notna guards & county[:5] != tract[:5]", "same guards + F.substring(c, 1, 5)"),
    "V08": ("real reason 1-9 vs action_taken in {3,7}; 10/blank on 3,7", "same logic with F.isin / F.coalesce"),
    "V09": ("profile_cltv_discrepancy = cltv - ltv_calc (diagnostic)", "same expression, no filter"),
    "V10": ("MD5 over chr(31)-joined 99 cols; groupby size/min(row_id)", "F.md5(F.concat_ws(chr(31), *cols)); Window.partitionBy(key).orderBy('row_id')"),
    "X05": ("state4() per field; precedence Present > No_Coapp > NA", "F.when chain with explicit isin sets; NULL-explicit race/eth"),
    "X09": ("sum of exempt tokens over 28 documented fields", "sum of F.when(cond, 1).otherwise(0)"),
}
# Step 2: Build the ledger table
def build_ledger():
    rows = []
    for key, e in LEDGER.items():
        base = key.split(":")[0].split(".")[0]
        py, sp = RULE_IMPL.get(base, ("", ""))
        rows.append({"key": key, "rule_id": base, "description": e["description"], "unit": e["unit"], "affected_raw": e["raw"],
                     "pct_raw": round(e["raw"] * 100.0 / TOTAL_RAW_ROWS, 4) if e["unit"] == "rows" else np.nan,
                     "affected_s0": e["s0"], "pct_s0": round(e["s0"] * 100.0 / WF["s0"], 4) if e["unit"] == "rows" else np.nan,
                     "pandas_impl": py, "pyspark_equiv": sp})
    for rid, desc, n in (("F01", "action_taken not in 1,2,3 (removed)", WF["raw"] - WF["f01"]),
                         ("F02", "business_or_commercial_purpose not in 2,1111 (removed)", WF["f01"] - WF["f02"]),
                         ("F03", "reverse_mortgage not in 2,1111 (removed)", WF["f02"] - WF["s0"])):
        rows.append({"key": rid, "rule_id": rid, "description": desc, "unit": "rows", "affected_raw": n,
                     "pct_raw": round(n * 100.0 / TOTAL_RAW_ROWS, 4), "affected_s0": 0, "pct_s0": 0.0,
                     "pandas_impl": "Series.isin([...]) positive membership", "pyspark_equiv": "df.filter(F.col(c).isin(...))"})
    return pd.DataFrame(rows).sort_values("key").reset_index(drop=True)

ledger_df = build_ledger()
save_and_log(ledger_df, "B06_rule_ledger.csv")
display(ledger_df[["key", "description", "unit", "affected_raw", "pct_raw", "affected_s0", "pct_s0"]].head(25))

## Section 6: Verification Against B05 Golden Numbers

- **Input:** `GOLDEN` (B05 targets) and the counts actually produced by this Pandas run (`LEDGER`, `WF`, `DUP_STATS`).
- **Process:**
  - Compare every golden metric with the actual value; show a table with `expected`, `actual`, `match`.
  - Second table: **expected-zero checks** — anomaly counters that the rulebook predicts to be 0 (invalid co-applicant states, non-zero tract values on tract-NA rows, ID length violations, V06–V08, ...).
  - If `ENFORCE_GOLDEN` is true, any mismatch stops the notebook (do not edit results to force agreement; investigate types, special codes, filters or duplicates).
- **Output:** `output/B06_golden_verification.csv`, `output/B06_expected_zero_checks.csv`.

In [ ]:
# Step 1: Actual values collected from this run
def L(key, col="raw"):
    return LEDGER[key][col] if key in LEDGER else 0

ACTUAL = {
    "raw_rows": WF["raw"], "f01_rows": WF["f01"], "f02_rows": WF["f02"], "s0_rows": WF["s0"], "s0_denied": WF["s0_sample_denied"],
    "f03_alone_on_f01": WF["f03_alone_on_f01"], "f02_f03_overlap": WF["f02_f03_overlap"],
    "f02_1111_in_s0": WF["f02_1111_in_s0"], "f03_1111_in_s0": WF["f03_1111_in_s0"],
    "v10_clusters": DUP_STATS["clusters"], "v10_extra_copies": L("V10:flag_dup_extra"),
    "dr1_is_1111_raw": L("X09:dr1_is_1111"), "t08_applicant_age_8888": L("T08:applicant_age_na"),
    "t08_co_age_8888": L("T08:co_applicant_age_na"), "t08_co_age_9999": L("T08:co_applicant_age_no_coapp"),
    "t05_lowercase_na_tract": L("T05:lowercase_na_census_tract"), "t05_non_msa_raw": L("T05:non_msa_99999"),
    "tract_null_raw": L("T12.0:tract_null"), "tract_null_s0": L("T12.0:tract_null", "s0"),
    "ffiec_no_geo_raw": L("T12.2:ffiec_zero_no_geo"), "ffiec_no_geo_s0": L("T12.2:ffiec_zero_no_geo", "s0"),
    "median_age_na_code_raw": L("T12.3:median_age_na_code"), "tract_income_zero_flag_raw": L("T12.4:tract_income_zero_flag"),
    "s5_all_rows": WF["s5_all_sample_rows"], "s5_all_denied": WF["s5_all_sample_denied"],
    "s5_keep_v02_rows": WF["s5_keep_v02_sample_rows"], "s5_keep_v02_denied": WF["s5_keep_v02_sample_denied"],
}
for k in ("V01", "V02", "V03", "V04", "V05"):
    ACTUAL[f"{k.lower()}_raw"], ACTUAL[f"{k.lower()}_s0"] = L(k), L(k, "s0")
for k in ("V06", "V07", "V08"):
    ACTUAL[f"{k.lower()}_raw"] = L(k)
for tag, key in (("x05_discordant", "X05:discordant"), ("x05_all_na", "X05:all_na"), ("x09_full", "X09:full_exempt"),
                 ("x09_partial", "X09:partial_exempt")):
    ACTUAL[f"{tag}_raw"], ACTUAL[f"{tag}_s0"] = L(key), L(key, "s0")

In [ ]:
# Step 1: Golden comparison table
gold_df = pd.DataFrame([{"metric": k, "expected": v, "actual": ACTUAL.get(k, np.nan)} for k, v in GOLDEN.items()])
gold_df["match"] = gold_df["expected"] == gold_df["actual"]
save_and_log(gold_df, "B06_golden_verification.csv")
display(gold_df[~gold_df["match"]] if (~gold_df["match"]).any() else gold_df.head(10))
print(f"Golden metrics matched: {int(gold_df['match'].sum())} / {len(gold_df)}")

# Step 2: Expected-zero anomaly counters
ZERO_KEYS = ["X05:invalid", "T08:applicant_age_invalid", "T08:applicant_age_blank", "T08:co_applicant_age_invalid",
             "T12.1:anomaly_nonzero_on_tract_na", "T12.2:anomaly_ffiec_zero_elsewhere", "T12.5:minority_out_of_range",
             "T05:length_violations", "V06", "V07", "V08", "V01"]
zero_df = pd.DataFrame([{"key": k, "affected_raw": L(k), "affected_s0": L(k, "s0")} for k in ZERO_KEYS])
zero_df["is_zero"] = (zero_df["affected_raw"] == 0) & (zero_df["affected_s0"] == 0)
save_and_log(zero_df, "B06_expected_zero_checks.csv")
display(zero_df)
if ENFORCE_GOLDEN:
    assert gold_df["match"].all(), "Golden mismatch: see B06_golden_verification.csv"
    assert zero_df["is_zero"].all(), "Expected-zero check failed: see B06_expected_zero_checks.csv"

## Section 7: Sample Waterfall and Racial Selection Waterfall (Plan B-7)

- **Input:** `WF` counters and the output file `B06_clean_all.parquet` (only the columns needed).
- **Process:**
  - **Sequential waterfall** Raw → F01 → F02 → F03 (= S0) with rows excluded and retention, plus a memo table (F03 alone, overlap, retained `1111` rows) — same layout as B05 so the two can be diffed.
  - **Racial selection waterfall:** application count and denial rate by `race_group` at four stages — F01, S0, S0 ∩ `fin_complete`, and S0 ∩ `fin_complete` ∩ valid `census_tract` (the RQ3 sample, Decision D-I). Shows the selection effect that Steps 3–4 must recompute.
- **Output:** `output/B06_waterfall_f01_f03.csv`, `output/B06_waterfall_memo.csv`, `output/B06_race_selection_waterfall.csv`.

In [ ]:
# Step 1: Sequential waterfall and memo table
wf_rows = [("Raw Applications", "All 2024 HMDA records", WF["raw"], 0),
           ("Rule F01", "action_taken IN ('1','2','3')", WF["f01"], WF["raw"] - WF["f01"]),
           ("Rule F02", "business_or_commercial_purpose IN ('2','1111')", WF["f02"], WF["f01"] - WF["f02"]),
           ("Rule F03 (S0)", "reverse_mortgage IN ('2','1111')", WF["s0"], WF["f02"] - WF["s0"])]
waterfall_df = pd.DataFrame(wf_rows, columns=["Step", "Criteria", "Applications", "Excluded"])
waterfall_df["Retention_of_Raw_Pct"] = (waterfall_df["Applications"] * 100.0 / WF["raw"]).round(2)
memo_df = pd.DataFrame([
    ("F03 exclusion alone on F01", WF["f03_alone_on_f01"]), ("F02 & F03 overlap on F01", WF["f02_f03_overlap"]),
    ("Combined F02 + F03 excluded", WF["f01"] - WF["s0"]),
    ("F02 code 1111 retained in S0", WF["f02_1111_in_s0"]), ("F03 code 1111 retained in S0", WF["f03_1111_in_s0"])],
    columns=["Metric", "Rows"])
memo_df["Pct_of_F01"] = (memo_df["Rows"] * 100.0 / WF["f01"]).round(4)
save_and_log(waterfall_df, "B06_waterfall_f01_f03.csv"); save_and_log(memo_df, "B06_waterfall_memo.csv")
display(waterfall_df); display(memo_df)
print(f"S0 denial rate: {WF['s0_sample_denied'] * 100.0 / WF['s0']:.2f}% ({WF['s0_sample_denied']:,} denied)")

In [ ]:
# Step 1: Load only the needed columns from clean_all
cols = ["race_group", "is_denied", "in_decision_sample", "in_s0", "fin_complete", "census_tract_status"]
tbl = pq.read_table(CLEAN_ALL_PATH, columns=cols)
d = tbl.drop_columns(["census_tract_status"]).to_pandas()
d["tract_ok"] = pc.equal(tbl["census_tract_status"], "value").to_pandas().to_numpy()
d["fin"] = d["in_s0"] & d["fin_complete"]
d["rq3"] = d["fin"] & d["tract_ok"]

# Step 2: Applications and denial rate by race at each stage
stages = {"f01": d["in_decision_sample"], "s0": d["in_s0"], "fin": d["fin"], "rq3": d["rq3"]}
parts = []
for name, mask in stages.items():
    g = d[mask].groupby("race_group", dropna=False)["is_denied"]
    parts.append(pd.DataFrame({f"{name}_apps": g.size(), f"{name}_denial_rate_pct": (g.mean() * 100).round(2)}))
race_wf_df = pd.concat(parts, axis=1).sort_values("s0_apps", ascending=False).reset_index()
race_wf_df["denial_shift_s0_to_fin_pp"] = (race_wf_df["fin_denial_rate_pct"] - race_wf_df["s0_denial_rate_pct"]).round(2)
assert race_wf_df["s0_apps"].sum() == WF["s0"], "Race table does not add up to S0"
save_and_log(race_wf_df, "B06_race_selection_waterfall.csv")
display(race_wf_df)
del d, tbl

## Section 8: Output Sanity Checks and V09 Diagnostic Profile

- **Input:** The two Parquet files and the B05 V09 profile (first liens in S0).
- **Process:**
  - Row-count and schema checks from Parquet metadata; `row_id` uniqueness in `clean_all`.
  - **No leftover tokens:** the five T05 ID columns must contain no `NA` / `Exempt` strings after cleaning.
  - **V09 profile (diagnostic, no exclusion):** for first liens in `analytic`, compute count, median and p95 of `|CLTV − LTV|` and compare with B05 cell 37 (tolerance 0.001 pp because B05 rounds to 3 decimals).
- **Output:** `output/B06_sanity_checks.csv`.

In [ ]:
# Step 1: Metadata, uniqueness and leftover-token checks
checks = []
meta_all, meta_an = pq.ParquetFile(CLEAN_ALL_PATH), pq.ParquetFile(ANALYTIC_PATH)
checks.append(("clean_all rows == raw rows", meta_all.metadata.num_rows == TOTAL_RAW_ROWS))
checks.append(("analytic rows == S0 rows", meta_an.metadata.num_rows == WF["s0"]))
checks.append(("identical column names", meta_all.schema_arrow.names == meta_an.schema_arrow.names))
checks.append(("row_id unique in clean_all", pq.read_table(CLEAN_ALL_PATH, columns=["row_id"])["row_id"].to_pandas().is_unique))
ids_tbl = pq.read_table(CLEAN_ALL_PATH, columns=list(ID_SPECS))
for c in ID_SPECS:
    left = pc.sum(pc.is_in(pc.utf8_upper(ids_tbl[c]), value_set=pa.array(["NA", "EXEMPT", ""]))).as_py() or 0
    checks.append((f"no NA/Exempt/blank tokens left in {c}", left == 0))
del ids_tbl

# Step 2: V09 first-lien profile vs B05 cell 37
an = pq.read_table(ANALYTIC_PATH, columns=["lien_status", "profile_cltv_discrepancy"]).to_pandas()
fl = an.loc[(an["lien_status"] == "1") & an["profile_cltv_discrepancy"].notna(), "profile_cltv_discrepancy"].abs()
v09 = {"count": len(fl), "median": round(fl.median(), 3), "p95": round(fl.quantile(0.95), 3)}
checks.append(("V09 first-lien count == B05 (5,033,106)", v09["count"] == 5_033_106))
checks.append(("V09 first-lien median within 0.001 of B05 (0.755)", abs(v09["median"] - 0.755) <= 0.001))
checks.append(("V09 first-lien p95 within 0.001 of B05 (4.710)", abs(v09["p95"] - 4.710) <= 0.001))
sanity_df = pd.DataFrame(checks, columns=["check", "passed"])
save_and_log(sanity_df, "B06_sanity_checks.csv")
display(sanity_df); print("V09 profile:", v09)
if ENFORCE_GOLDEN:
    assert sanity_df["passed"].all(), "Sanity check failed: see B06_sanity_checks.csv"
else:
    assert sanity_df["passed"][:4].all(), "Structural sanity check failed"

## Section 9: Milestone Summary and Hand-off to B07 (PySpark)

- **Input:** Counters and tables produced above.
- **Process:** Assemble the summary dynamically from Python variables (no typed-in numbers, Convention 4.3) and list every artifact with its path, size and shape for the B07 / B08 hand-off.
- **Output:** `output/B06_summary.csv`.

In [ ]:
# Step 1: Dynamic milestone summary
n_match, n_gold = int(gold_df["match"].sum()), len(gold_df)
summary_df = pd.DataFrame([
    ("clean_all", f"{PIPE['rows_clean_all']:,} rows x {PIPE['n_cols']} cols (raw rows kept)", CLEAN_ALL_PATH.name),
    ("analytic (S0)", f"{PIPE['rows_analytic']:,} rows; denial rate {WF['s0_sample_denied'] * 100.0 / WF['s0']:.2f}%", ANALYTIC_PATH.name),
    ("Golden verification vs B05", f"{n_match} / {n_gold} metrics matched", "B06_golden_verification.csv"),
    ("Expected-zero checks", f"{int(zero_df['is_zero'].sum())} / {len(zero_df)} are zero", "B06_expected_zero_checks.csv"),
    ("Duplicates (V10)", f"{DUP_STATS['clusters']:,} clusters, {DUP_STATS['extra_copies']:,} extra copies", "B06_dup_table.csv"),
    ("S5 samples", f"S5_all {WF['s5_all_sample_rows']:,} rows; S5_keep_v02 {WF['s5_keep_v02_sample_rows']:,} rows", "B06_rule_ledger.csv"),
    ("Pipeline time", f"{PIPE['seconds'] / 60:.1f} min (Pandas, chunk {CHUNK_ROWS:,} rows)", "-"),
], columns=["Item", "Finding", "Artifact"])
save_and_log(summary_df, "B06_summary.csv")
display(summary_df)
print("[SUCCESS] Task B06 completed. Next: B07 (PySpark) reuses B06_s4_winsor_thresholds.json, B06_lender_sizes.csv and the ledger's pyspark_equiv column.")